# 第5部分：规则提取（从LGBM树中挖掘策略规则）

**目的：** 把LGBM学到的"判断逻辑"翻译成人能读的策略规则

## 通俗理解

LGBM训练出了200棵决策树，每棵树都是一串IF-THEN判断：

```
IF 信用分<=450 AND 近3月查询>5次
THEN 这人大概率违约
```

我们要从这几万条路径中，找出**又准又覆盖面广**的策略规则。

## 输出示例

| 规则 | 命中样本 | 覆盖率 | 坏账率 | 提升度 |
|------|---------|--------|--------|--------|
| credit_score<=450 AND query_3m>5 | 3000 | 6% | 15% | 3.0x |
| debt_ratio>0.6 AND income<5000 | 2500 | 5% | 12% | 2.4x |

提升度3.0x = 这群人的坏账率是平均的3倍！可以作为拒绝规则。

In [ ]:
import pandas as pd
import numpy as np

## 步骤1：从LGBM模型中提取决策路径

### 原理图解

```
一棵决策树长这样：

              [信用分<=500?]
             /              \
      [查询次数>3?]       [收入>10000?]
      /        \           /         \
    叶子A    叶子B      叶子C       叶子D

叶子A的路径 = "信用分<=500 AND 查询次数>3"
叶子B的路径 = "信用分<=500 AND 查询次数<=3"
叶子C的路径 = "信用分>500 AND 收入>10000"
叶子D的路径 = "信用分>500 AND 收入<=10000"
```

我们要做的：把200棵树的所有叶子路径都提取出来，统计哪些路径出现最频繁

In [ ]:
def extract_decision_paths_fast(model, feature_names, sample_data, max_depth=3):
    """
    从LGBM的每棵树中提取决策路径（规则）

    参数：
        model: 训练好的LGBM模型
        feature_names: 特征名列表
        sample_data: 样本数据（用来看每个样本走了哪条路径）
        max_depth: 最多追溯几层
            3层 = 最多3个条件组合 (A AND B AND C)
            4层 = 最多4个条件

    返回：
        Series: 规则字符串 -> 出现频次
    """
    # pred_leaf=True: 预测每个样本落在每棵树的哪个叶子上
    # 返回形状: (样本数, 树的棵数)
    leaf_ids = model.predict(sample_data, pred_leaf=True)

    # dump_model(): 把模型的树结构导出为Python字典
    tree_dicts = model.dump_model()['tree_info']

    # ===== 第一步：遍历每棵树，建立 "叶子ID -> 路径" 的映射 =====
    tree_paths = []

    for tree_idx in range(len(tree_dicts)):
        tree_struct = tree_dicts[tree_idx]['tree_structure']
        node_paths = {}  # {叶子ID: [条件1, 条件2, ...]}

        # 用"栈"遍历树（比递归更快、不会栈溢出）
        # 栈里每个元素 = (当前节点, 到达该节点的路径, 当前深度)
        stack = [(tree_struct, [], 0)]

        while stack:
            node, current_path, depth = stack.pop()

            # 到达叶子 或 超过最大深度: 记录这条路径
            if depth >= max_depth or 'split_feature' not in node:
                if 'leaf_index' in node:
                    node_paths[node['leaf_index']] = current_path
                continue

            # 当前节点的分裂信息
            feat_idx = node['split_feature']        # 用哪个特征分裂
            threshold = node['threshold']           # 阈值是多少
            feat_name = feature_names[feat_idx]     # 特征名

            # 构造人可读的条件
            if isinstance(threshold, (int, float)):
                left_cond = f"{feat_name}<={threshold:.2f}"
                right_cond = f"{feat_name}>{threshold:.2f}"
            else:
                left_cond = f"{feat_name}<={threshold}"
                right_cond = f"{feat_name}>{threshold}"

            # 把左右子树压入栈继续遍历
            if 'right_child' in node:
                stack.append((node['right_child'], current_path + [right_cond], depth + 1))
            if 'left_child' in node:
                stack.append((node['left_child'], current_path + [left_cond], depth + 1))

        tree_paths.append(node_paths)

    # ===== 第二步：统计每条规则被多少样本"走过" =====
    rules = []
    for tree_idx in range(leaf_ids.shape[1]):       # 遍历每棵树
        for sample_idx in range(len(sample_data)):  # 遍历每个样本
            leaf_id = leaf_ids[sample_idx, tree_idx]
            path = tree_paths[tree_idx].get(leaf_id, [])
            if path:
                rules.append(' AND '.join(path))  # 拼成规则字符串

    # 按频次排序（出现越多 = 模型越依赖这条规则）
    return pd.Series(rules).value_counts()

## 步骤2：在原始数据上验证规则效果

提取出来的规则只是"模型认为重要的路径"，还需要验证：
1. 这条规则能命中多少人？（覆盖率）
2. 命中的人坏账率高吗？（精准度）
3. 比平均高多少？（提升度）

In [ ]:
def safe_eval_rule(df, rule_str, label_col='dob4_ever10_flg'):
    """
    把规则字符串应用到数据上
    返回：True/False的Series（True=这个人满足规则条件）

    为什么叫"safe"？
    树里提取的规则用特殊符号，pandas需要标准符号，需要格式转换和异常处理
    """
    try:
        # 符号标准化
        normalized = (
            rule_str.replace('||', '|')
                    .strip()
        )

        # 拆分AND条件，逐个转换
        conditions = []
        for raw_condition in normalized.split(' AND '):
            condition = raw_condition.strip()

            if '<=' in condition:
                parts = [p.strip() for p in condition.split('<=') if p.strip()]
                if len(parts) == 2:
                    conditions.append(f"`{parts[0]}`<={parts[1]}")
            elif '>=' in condition:
                parts = [p.strip() for p in condition.split('>=') if p.strip()]
                if len(parts) == 2:
                    conditions.append(f"`{parts[0]}`>={parts[1]}")
            elif '>' in condition:
                parts = [p.strip() for p in condition.split('>') if p.strip()]
                if len(parts) == 2:
                    conditions.append(f"`{parts[0]}`>{parts[1]}")
            elif '<' in condition:
                parts = [p.strip() for p in condition.split('<') if p.strip()]
                if len(parts) == 2:
                    conditions.append(f"`{parts[0]}`<{parts[1]}")

        # 用 & 连接（pandas的AND）
        safe_expr = ' & '.join(conditions)
        return df.eval(safe_expr, engine='python')

    except Exception as e:
        return pd.Series(False, index=df.index)

In [ ]:
def evaluate_rules(df, rules, label_col='dob4_ever10_flg',
                   min_coverage=0.05, max_coverage=0.2):
    """
    批量评估规则效果

    对每条规则计算：
        命中样本数: 这条规则能抓住多少人
        覆盖率: 命中人数 / 总人数
        坏账率: 命中的人中有多少违约
        提升度: 坏账率 / 大盘平均坏账率

    为什么限制覆盖率？
        太低(<5%): 太苛刻，只抓住少数人，对大盘影响小
        太高(>20%): 太宽泛，什么人都命中，没有区分力
    """
    results = []
    base_bad_rate = df[label_col].mean()  # 大盘平均坏账率

    for rule in rules:
        mask = safe_eval_rule(df, rule, label_col)
        if mask.sum() == 0:
            continue

        coverage = mask.mean()

        # 覆盖率不在合理范围内的跳过
        if not (min_coverage <= coverage <= max_coverage):
            continue

        bad_rate = df[label_col][mask].mean()
        bads = df[label_col][mask].sum()

        results.append({
            'rule': rule,
            'hit_count': mask.sum(),
            'bad_count': bads,
            'coverage': f"{coverage:.2%}",
            'bad_rate': f"{bad_rate:.2%}",
            'lift': f"{bad_rate/base_bad_rate:.2f}x",
        })

    return pd.DataFrame(results).sort_values('bad_rate', ascending=False)

## 步骤3：过滤业务不合理的规则

模型可能产生"统计上正确但业务上荒谬"的规则：

- `credit_score>700` 命中的人坏账率高？不合理！高分应该低风险
- `query_count<1` 命中的人坏账率高？不合理！查询少应该低风险

需要用业务知识过滤掉这些"统计幻觉"

In [ ]:
def get_default_good_vars(df):
    """
    找出所有"模型评分"类的特征
    （变量名同时包含model和score的）

    这些特征：分数越高 -> 风险越低
    所以合理规则应该是 "score<=X" （分低=危险）
    而不是 "score>X" （分高=危险，违反逻辑！）
    """
    good_vars = []
    for col in df.columns:
        col_lower = col.lower()
        if 'model' in col_lower and 'score' in col_lower:
            good_vars.append(col)
    return good_vars


def should_exclude_rule(rule_str, good_vars=None, bad_vars=None):
    """
    判断规则是否业务不合理

    排除逻辑：
        good_vars（分高=好）出现 ">" 条件 -> 排除
        bad_vars（值大=坏）出现 "<" 条件 -> 排除
    """
    if not good_vars and not bad_vars:
        return False

    normalized = rule_str.strip()

    for raw_condition in normalized.split(' AND '):
        condition = raw_condition.strip()

        # good_vars出现">": 说高分危险？不合理！排除
        if good_vars:
            if '>' in condition and '<' not in condition:
                parts = [p.strip() for p in condition.split('>') if p.strip()]
                if len(parts) == 2 and parts[0] in good_vars:
                    return True
            elif '>=' in condition:
                parts = [p.strip() for p in condition.split('>=') if p.strip()]
                if len(parts) == 2 and parts[0] in good_vars:
                    return True

        # bad_vars出现"<": 说查询少危险？不合理！排除
        if bad_vars:
            if '<' in condition and '>' not in condition:
                parts = [p.strip() for p in condition.split('<') if p.strip()]
                if len(parts) == 2 and parts[0] in bad_vars:
                    return True
            elif '<=' in condition:
                parts = [p.strip() for p in condition.split('<=') if p.strip()]
                if len(parts) == 2 and parts[0] in bad_vars:
                    return True

    return False  # 合理，保留

## 完整的规则提取流程

In [ ]:
# 第1步：从模型中提取所有规则
top_rules = extract_decision_paths_fast(
    lgbmodel,
    feature_names=train_data_x.columns.tolist(),
    sample_data=train_data_x,
    max_depth=4  # 最多4个条件
)

print(f"共提取到 {len(top_rules)} 条不重复规则")
print(f"\n出现频次最高的5条规则：")
top_rules.head()

In [ ]:
# 第2步：评估规则效果
rule_results = evaluate_rules(
    df=dftmp,
    rules=top_rules.index[:500],  # 取频次最高的前500条评估
    label_col='dob4_ever10_flg',
    min_coverage=0.05,   # 最少覆盖5%
    max_coverage=0.2     # 最多覆盖20%
)

print(f"有效规则数: {len(rule_results)}")
print(f"\nTop 10 高风险规则：")
rule_results.head(10)

---
### 面试考点

| 问题 | 答案 |
|------|------|
| 提升度2.5x什么意思？ | 命中人群的坏账率是平均的2.5倍 |
| 覆盖率为什么限制5%-20%？ | <5%抓的人太少没意义，>20%太宽泛没区分力 |
| 为什么要过滤"score>X"的规则？ | 评分越高应该越安全，说高分危险违反业务逻辑 |
| 规则太多怎么选？ | 1.提升度高 2.覆盖率适中 3.业务可解释 4.规则间重叠度低 |
| 这些规则怎么上线？ | 写成IF-THEN逻辑部署到决策引擎，命中即拒绝/加审 |